In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [24]:
from langchain_mistralai import MistralAIEmbeddings,ChatMistralAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import time 


In [3]:
llm_model = ChatMistralAI(
    model = "mistral-small-2603"
)
embedding_model = MistralAIEmbeddings(
)

In [21]:
from email_validator.syntax import split_email
def generate_hypo_docs(query):
    prompt=f"Generate a hypothetical best-guess answer to: {query}, only the answer of the query is required."
    response = llm_model.invoke(prompt).content
    return response 

def embed_response(response):
    embedded_response = embedding_model.embed_query(response)
    return embedded_response

def load_documents(file_path =r"d:\My-Learning\AdvanceRag\Documents\Indian_constitution.pdf"):
    loader = PyPDFLoader(file_path) 
    documents = loader.load()
    return documents

def text_split(document):
    splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=100)
    chunks = splitter.split_documents(document)
    return chunks

def create_vector_db(embedding_model,store_dir=r"d:\My-Learning\AdvanceRag\Documents\Vector_db"):
    vector_store = Chroma(
        collection_name = "Documents",
        embedding_function = embedding_model,
        persist_directory = store_dir 
    )
    return vector_store

def insert_vector_db(vector_store,chunks):
    try:
        vector_store.add_documents(chunks)
        print("data has been added successfully")
    except:
        print("an error has occured")

def search_vector_db(vector_store,response):
    start_time = time.time()
    retriver = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)
    reterived_docs = retriver.invoke(response)
    context = "\n\n".join(docs.page_content for docs in reterived_docs)
    endtime = time.time()
    print(f"time taken to reterive the documents : {endtime-start_time}")
    return context


In [22]:
def workflow_store():
    # load file into documents 
    documents = load_documents()
    # split into text
    chunks = text_split(documents)
    # create vector database 
    vector_db = create_vector_db(embedding_model)
    # insert data into vector database 
    insert_vector_db(vector_db,chunks)
    print("Workflow completed succesfully.")
    return vector_db

def retrive_data(query,vector_db):
    #generate hypothetical
    start_time = time.time()
    response = generate_hypo_docs(query)
    # print(response)
    # search the embeded response in the vectordb 
    context = search_vector_db(vector_db,response)
    # print(context)
    prompt = f"""
    You are a helpful assistant.
    Your main goal is to answer the user's QUERY from PROVIDED CONTEXT,
    INSTRUCTION : 
    1. analyse the user query
    2. analyse the provided context and talior a concised and simple response for the user.
    3. if the CONTEXT isnt appropriate or isnt related to query just say "I dont know".

    QUERY : {query}
    CONTEXT : {context}
    """
    final_output = llm_model.invoke(prompt)
    print(final_output.content)
    end_time = time.time()
    print(f"time taken to complete the Reterival section : {end_time-start_time}")

In [8]:
vector_store = workflow_store()

data has been added successfully
Workflow completed succesfully.


In [31]:
query = input("ask your question from constitution of India")
retrive_data(query,vector_store)

time taken to reterive the documents : 0.7110364437103271
The Indian Constitution is divided into **25 parts**.

(Note: While the provided context only shows parts of the Constitution, the standard structure includes 25 parts, 12 schedules, and the preamble.)
time taken to complete the Reterival section : 36.61701798439026
